# v042_stage2_m3 — M3's feature groups in stage 2 of the two-stage matcher, on the mock

| Field | Value |
|---|---|
| **Version** | `v042_stage2_m3` |
| **Plan group** | C2 (with C3–C5): M3's feature groups in the two-stage pipeline |
| **Parent version** | `v107` (`v107_tight_rule`, est_public 0.9659); same-machine baseline: arm A below |
| **Author** | M3 |
| **Date** | 2026-09-26 |
| **Status** | kept |
| **Hypothesis** | M3's idf, token_freq, ctx_idf and address_extra groups, computed on the kept pairs and added to stage 2, raise est_public on the mock by ≥ +0.001 over the same pipeline without them. |

v040 showed M3's four groups add +0.0026 plain-val F0.5 to a single-stage model. Since then the
team moved to a two-stage matcher judged on the test-shaped mock (TRACKER "Mock-test
protocol", "Tight mock"): v101 scores every candidate (stage 1), the 16 best per S1 with
p1 ≥ 0.01 stay, and an XGBoost stage 2 learns from v101's pair features plus competition and
anchor features, trained at test density on the mock's fit entities; the rule is tuned for the
tight score (false merges ×1.45), whose shift gives **est_public**, the decision number.

This version adds M3's groups to stage 2 only (`TwoStageConfig.extra_groups`: built on the ~5
kept pairs per S1, with pool statistics of the whole partition). One stage-1 pass serves three
stage-2 arms, so their differences are the features' alone:

```
v101 (stage 1) -> filter -> competition + anchors + M3 groups -> stage 2 arms:
   A  v104's columns (no M3)             same-machine reproduction of v104 / v107
   B  + all four M3 groups               this version
   C  + M3 groups without token_freq     the raw pool counts' share (test pools are bigger)
-> v107's rule tuning (tight threshold vs tight expected-F0.5) -> est_public on mock val
```

## 1. Hypothesis

* **Change vs parent:** stage 2 also reads M3's 23 columns (`idf` 8, `token_freq` 4, `ctx_idf` 5,
  `address_extra` 6) on the kept pairs. Stage 1, the filter, the stage-2 model settings, the
  mock and the rule tuning are v104's / v107's.
* **Why it should help:** stage 2 ranks a handful of look-alike candidates; v101's pair
  features say little about which of two same-name records is the one (v104's stage 2 put
  72 % of its gain on `pool_gap`). M3's groups add what v040 found useful: rare shared tokens
  (idf), house-number containment (address_extra, v040's strongest false-merge separator), and
  how common the exact name is in the pool (token_freq).
* **Prediction (falsifiable):** est_public(B) ≥ est_public(A) + 0.001 on the mock val entities;
  mock F0.5 up; singletons not down by more than 0.002.
* **Risk:** token_freq counts shift with pool size; the mock has the test's density, so arm C
  measures their share where it matters.
* **Discard if:** est_public(B) ≤ est_public(A).

## 2. Setup

Imports first; the cell after holds every version-specific value.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import apply_rule, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS, FEATURE_COLUMNS, feature_names
from entity_resolution.mock import FP_WEIGHT, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, mem_guard, mock_scores, peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1, run_test_two_stage,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

### Parameters of this version

Stage 1 is v101 (its fitted pipeline in `experiments/v101_name_frequency/artifacts/`, re-fitted
deterministically on this machine where the folder was empty). The stage-2 settings are
v104's, read from its `metrics.json`, plus `extra_groups`.

In [2]:
VERSION = "v042_stage2_m3"
PARENT = "v107_tight_rule"          # logged decision baseline (est_public 0.9659)
GROUP = "C2"
OWNER = "M3"
M3_GROUPS = ("idf", "token_freq", "ctx_idf", "address_extra")
RUN_TEST = False                    # test inference only for shortlisted versions (§9)

cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000))          # v101 = stage 1

Derived paths, stage 1, the stage-2 configuration and the parent's logged scores. The stage-1
cache directory is specific to this combination (v101, blocking key, filter, anchors, M3
groups), as `twostage._cached_stage1` requires.

In [3]:
EXP_DIR = C.EXPERIMENTS / VERSION
ARTIFACTS = EXP_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
stage1 = Fitted.load(C.EXPERIMENTS / "v101_name_frequency" / "artifacts", cfg)
rec = json.loads((C.EXPERIMENTS / "v104_two_stage" / "metrics.json").read_text())["metrics"][
    "two_stage"]
tcfg = TwoStageConfig(**{**rec, "model": MatcherParams(**rec["model"]),
                         "train_roles": tuple(rec.get("train_roles", ("fit",))),
                         "extra_groups": M3_GROUPS})
STAGE1_CACHE = (cfg.cache_dir / "stage1" / f"v101_{cfg.blocking.key()}_f{tcfg.floor}"
                f"_k{tcfg.max_cands}_a{int(tcfg.anchors)}_m3")
parent = json.loads((C.EXPERIMENTS / PARENT / "metrics.json").read_text())
parent_scores = parent["metrics"]["comparison"]["v107 tight rule"]
m3_columns = feature_names(M3_GROUPS)
timings: dict[str, float] = {}
t_start = time.time()
print(f"stage 1 rule {stage1.rule}; stage-1 features {len(stage1.matcher.feature_names_)}; "
      f"M3 columns {len(m3_columns)}")
print("two-stage:", json.dumps(tcfg.record())[:400])
print("parent v107:", {k: round(v, 5) for k, v in parent_scores.items()})

stage 1 rule DecisionRule(tau_abs=0.42, tau_rel=0.0, tau_single=0.52, max_matches=11, one_to_one=True); stage-1 features 53; M3 columns 23
two-stage: {"floor": 0.01, "max_cands": 16, "folds": 2, "seed": 6161, "n_stop_s1": 50000, "anchors": true, "cohesion": false, "rivals": false, "train_roles": ["fit"], "model": {"backend": "xgb", "num_leaves": 63, "learning_rate": 0.05, "n_estimators": 4000, "early_stopping": 100, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "min_data_in_leaf": 200, "lambda_l2": 1.0, "max_bin": 255, "s
parent v107: {'f_beta': 0.97451, 'f_tight': 0.97308, 'est_public': 0.96588, 'f_beta_singletons': 0.98367, 'pair_precision': 0.99648, 'pair_recall': 0.93423}


## 3. Data: the mock fold

Built exactly as in v104 / v107: per country, the test's pool size and pool records per S1,
v101's training sample dropped first, val / tune / fit roles by id hash.

In [4]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
del train, val, fit_fold, tune_fold, fit_sample
display(mock.info)
mock.fold.summary()

,country,s1_train,pool_train,keep_frac,s1_kept,pool_kept,s1_present,pool_per_s1,test_pool_per_s1,present_val,present_tune,present_fit
0,India,883188,4133346,1.000,883188,4133346,709678,5.824,5.824,176522,176208,356948
1,US,1323633,6186873,0.617,815850,3816702,663049,5.756,5.756,163562,163325,336162


{'fold': 'mock',
 's1': 1372727,
 's2': 3878856,
 's3': 4071192,
 'true_pairs': 4753992,
 'singleton_share': 0.0558}

## 4. Method

### 4.1 Stage 1 and the candidate filter (as v104)

v101 scores every blocked pair of each mock country; the 16 best per S1 with p1 ≥ 0.01 stay.
Competition features use every pair, anchors and M3's groups the kept ones. The filter report
shows the candidate recall that stage 2 inherits.

In [5]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_seconds"] = round(time.time() - t0, 2)
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
print(f"stage 1 {timings['stage1_seconds']:.0f} s; pairs {sum(o.n_all for o in outs.values()):,}"
      f" -> kept {len(kept):,}; frame columns {next(iter(outs.values())).X.shape[1]}")
filter_report = pd.DataFrame({r: blocking_report(kept, mock.part(r))
                              for r in ("fit", "tune", "val")}).T
del kept
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean",
               "candidates_p95"]]

stage 1 2964 s; pairs 47,348,383 -> kept 6,311,331; frame columns 93


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
fit,0.964196,0.996389,0.987477,4.601084,9.0
tune,0.964089,0.996298,0.987441,4.595480,9.0
val,0.964386,0.996392,0.987585,4.592856,9.0


### 4.2 Three stage-2 arms and v107's rule tuning

Each arm trains v104's cross-fitted XGBoost stage 2 on its columns, scores the mock's tune and
val entities (1-to-1 over every present entity), then tunes v107's two rule families on the
tune entities for the tight score (threshold grid and expected-F0.5 decoding, false merges
×1.45) and keeps the better; `mock_scores` scores the val entities with it.

In [6]:
all_cols = list(next(iter(outs.values())).X.columns)
ARMS = {
    "A: v104 columns": [c for c in all_cols if c not in m3_columns],
    "B: + M3 groups": all_cols,
    "C: + M3 without token_freq": [c for c in all_cols
                                   if c not in set(FEATURE_COLUMNS["token_freq"])],
}
TUNE_KW = {"gammas": (0.7, 0.85, 1.0, 1.2, 1.5, 2.0), "misses": (0.0, 0.05, 0.1, 0.2, 0.4),
           "fp_weight": FP_WEIGHT}
SCORE_COLS = ["f_beta", "f_tight", "est_public", "f_beta_singletons", "f_beta_matched",
              "pair_precision", "pair_recall"]


def run_arm(columns: list[str]) -> dict:
    """Stage 2 on ``columns``, mock scores, v107's rule choice and the val scores.

    Returns the models, their fit info, the scored tune + val pairs, the chosen rule with its
    tuning table, ``mock_scores`` of the val entities, the mean gain share per feature and the
    seconds taken.
    """
    t0 = time.time()
    models, info = fit_stage2(outs, mock, tcfg, columns=columns)
    scored, _ = mock_scored(outs, models, mock, tcfg)
    rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
    tune_part = mock.part("tune")
    rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
    rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                    **TUNE_KW)
    best_t, best_e = float(table_t["f_beta"].max()), float(table_e["f_beta"].max())
    rule, table = (rule_e, table_e) if best_e > best_t else (rule_t, table_t)
    importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
    return {"models": models, "info": info, "scored": scored, "rule": rule, "table": table,
            "res": mock_scores(scored, mock, rule), "tune_threshold": best_t,
            "tune_expected": best_e, "seconds": round(time.time() - t0, 1),
            "importance": importance.sort_values(ascending=False)}

## 5. Evaluation on the mock

### 5.1 The arms

Each arm takes a few minutes of GPU training (two cross-fitted models) plus the rule tuning.

In [7]:
arms: dict[str, dict] = {}
for name, cols in ARMS.items():
    arms[name] = run_arm(cols)
    r = arms[name]["res"].loc["all"]
    print(f"{name:28s} {len(cols):3d} cols {arms[name]['seconds']:6.0f} s  "
          f"est_public {r['est_public']:.5f}  mock F0.5 {r['f_beta']:.5f}  "
          f"rule {arms[name]['rule']}")
    mem_guard(name)
timings["arms_seconds"] = round(sum(a["seconds"] for a in arms.values()), 1)

A: v104 columns               70 cols    423 s  est_public 0.96588  mock F0.5 0.97451  rule ExpectedRule(gamma=1.5, miss=0.05, max_matches=11, one_to_one=True)


B: + M3 groups                93 cols    488 s  est_public 0.96788  mock F0.5 0.97623  rule ExpectedRule(gamma=1.5, miss=0.0, max_matches=11, one_to_one=True)


C: + M3 without token_freq    89 cols    497 s  est_public 0.96751  mock F0.5 0.97601  rule ExpectedRule(gamma=1.5, miss=0.4, max_matches=11, one_to_one=True)


### 5.2 Arms against each other and against the logged v107

`est_public` = tight mock F0.5 − 0.0072 (`mock.PUBLIC_OFFSET`), calibrated so v101 / v103 give
their public 0.955 / 0.961. Arm A is the same pipeline as v104 + v107 re-run on this machine; it
should land close to v107's logged numbers (XGBoost on another GPU, same seeds).

In [8]:
table = pd.DataFrame({name: a["res"].loc["all", SCORE_COLS] for name, a in arms.items()}).T
table.loc["v107 (logged)"] = pd.Series(parent_scores).reindex(SCORE_COLS)
table = table.astype(float)
base_est = table.loc["A: v104 columns", "est_public"]
table["d_est_vs_A"] = table["est_public"] - base_est
table["d_f05_vs_A"] = table["f_beta"] - table.loc["A: v104 columns", "f_beta"]
display(table.round(5))
by_country = pd.concat({name: a["res"].drop(index="all")[["f_beta", "est_public",
                                                          "f_beta_singletons"]]
                        for name, a in arms.items()})
by_country.round(5)

,f_beta,f_tight,est_public,f_beta_singletons,f_beta_matched,pair_precision,pair_recall,d_est_vs_A,d_f05_vs_A
A: v104 columns,0.97451,0.97308,0.96588,0.98367,0.97397,0.99648,0.93423,0.00000,0.00000
B: + M3 groups,0.97623,0.97508,0.96788,0.98655,0.97562,0.99726,0.93802,0.00200,0.00172
C: + M3 without token_freq,0.97601,0.97471,0.96751,0.98414,0.97552,0.99692,0.93760,0.00163,0.00150
v107 (logged),0.97451,0.97308,0.96588,0.98367,NaN,0.99648,0.93423,0.00000,0.00000


f_beta  est_public  f_beta_singletons
A: v104 columns            India  0.97014     0.96141            0.98145
                           US     0.97923     0.97070            0.98607
B: + M3 groups             India  0.97216     0.96377            0.98559
                           US     0.98062     0.97232            0.98759
C: + M3 without token_freq India  0.97198     0.96343            0.98256
                           US     0.98036     0.97192            0.98585

### 5.3 What stage 2 reads from M3's groups

Gain share of every column in arm B (mean of the two cross-fitted models), M3's columns flagged,
and each M3 group's total.

In [9]:
imp = arms["B: + M3 groups"]["importance"]
top = imp.head(25).rename("gain share").to_frame()
top["M3"] = top.index.isin(m3_columns)
display(top)
m3_imp = pd.DataFrame({"gain": imp.reindex(m3_columns).fillna(0.0),
                       "rank": imp.rank(ascending=False).reindex(m3_columns)})
group_share = {g: float(imp.reindex(FEATURE_COLUMNS[g]).fillna(0).sum()) for g in M3_GROUPS}
print("M3 share of stage-2 gain:", round(float(m3_imp["gain"].sum()), 4), group_share)
m3_imp.sort_values("gain", ascending=False)

,gain share,M3
feature,,
pool_gap,0.712773,False
p1,0.193251,False
s1_gap,0.012459,False
pool_best_other,0.005600,False
pool_p1_sum,0.005179,False
s1_p1_sum,0.004143,False
freq_addr_r,0.003856,True
idf_name_cover_r,0.003022,True
nm_token_set,0.001997,False


M3 share of stage-2 gain: 0.0238 {'idf': 0.012507969886186136, 'token_freq': 0.005141841223541178, 'ctx_idf': 0.00257897797238579, 'address_extra': 0.0036188329575547315}


,gain,rank
feature,,
freq_addr_r,3.856407e-03,7.0
idf_name_cover_r,3.021697e-03,8.0
idf_name_top,1.878727e-03,10.0
idf_addr_top,1.811727e-03,12.0
idf_addr_cos,1.749338e-03,13.0
idf_name_cover_l,1.217382e-03,22.0
num_contain_r,1.214801e-03,23.0
idf_addr_cover_r,1.043994e-03,30.0
num_contain_l,1.009878e-03,31.0


## 6. Error analysis

### 6.1 Error kinds, arm A against the best arm

Counts on the mock val entities under each arm's own rule.

In [10]:
KINDS = ("false_merge", "missed", "false_singleton", "singleton_merge")
part = mock.part("val")
val_ids = pd.Index(part.s1[C.ENTITY_ID])
best_name = max(arms, key=lambda n: arms[n]["res"].loc["all", "est_public"])
matches: dict[str, pd.DataFrame] = {}
counts: dict[str, dict] = {}
for name in dict.fromkeys(["A: v104 columns", "B: + M3 groups", best_name]):
    a = arms[name]
    matches[name] = apply_rule(a["scored"][isin(a["scored"][C.S1_ID], val_ids)], a["rule"])
    counts[name] = {k: len(error_samples(matches[name], part, k, n=10**9)) for k in KINDS}
print("best arm:", best_name)
pd.DataFrame(counts)

best arm: B: + M3 groups


,A: v104 columns,B: + M3 groups
false_merge,3522,2744
missed,74223,69866
false_singleton,3154,3063
singleton_merge,362,289


### 6.2 Pairs fixed and broken by M3's groups (arm A → arm B)

A pair is identified by its two ids; truth is the mock val entities' true pairs.

In [11]:
def keys(df: pd.DataFrame) -> set[str]:
    """``source1_entity_id|entity_id`` of every row (sets of pairs)."""
    return set((df[C.S1_ID].astype(str) + "|" + df[C.ENTITY_ID].astype(str)).tolist())


truth = keys(part.pairs)
a_s, b_s = keys(matches["A: v104 columns"]), keys(matches["B: + M3 groups"])
transitions = {
    "fixed: false merge dropped": len({k for k in a_s - b_s if k not in truth}),
    "fixed: miss now predicted": len({k for k in b_s - a_s if k in truth}),
    "broken: new false merge": len({k for k in b_s - a_s if k not in truth}),
    "broken: true pair lost": len({k for k in a_s - b_s if k in truth}),
}
pd.Series(transitions, name="pairs").to_frame()

,pairs
fixed: false merge dropped,1247
fixed: miss now predicted,6494
broken: new false merge,396
broken: true pair lost,2046


### 6.3 M3's error categories (doc 18 §3) on arm B's errors

The categories M3 owns, tagged from the stage-2 frame of each error pair (first true condition
wins, as in doc 18): misses that are candidates (`name_typo`, `word_order`, `dba_trade_name`,
`missing_address_component`, `postal_code_mismatch`) and false merges
(`same_name_different_business`, `same_address_different_business`). Misses outside the kept
candidates are blocking or filter losses (M1's).

In [12]:
NEED = ["core_ratio", "tok_len_l", "tok_len_r", "sorted_eq", "tok_jaccard", "non_latin_r",
        "ad_contain", "ad_contain_r", "addr_len_ratio", "postcode_eq", "core_token_set",
        "ad_jaccard", "ad_token_set"]
frame = pd.concat([pd.concat([o.pairs, o.X[NEED]], axis=1)[isin(o.pairs[C.S1_ID], val_ids)]
                   for o in outs.values()], ignore_index=True)


def tag(df: pd.DataFrame, kind: str) -> pd.Series:
    """Doc-18 category of each error pair of ``kind`` (missed or false_merge)."""
    f = df.merge(frame, on=[C.S1_ID, C.ENTITY_ID], how="left")
    cand = f["core_ratio"].notna() | f["ad_token_set"].notna() | f["tok_len_l"].notna()
    if kind == "missed":
        rules = [
            ("not a kept candidate (blocking / filter)", ~cand),
            ("name_typo", (f["core_ratio"] >= 0.6) & (f["core_ratio"] < 0.95)
             & (f["tok_len_l"] == f["tok_len_r"])),
            ("word_order", (f["sorted_eq"] == 1) & (f["core_ratio"] < 1)),
            ("dba_trade_name", (f["tok_jaccard"] == 0) & (f["non_latin_r"] == 0)),
            ("missing_address_component",
             (np.fmax(f["ad_contain"], f["ad_contain_r"]) >= 0.9) & (f["addr_len_ratio"] < 0.8)),
            ("postal_code_mismatch", f["postcode_eq"] == 0),
        ]
    else:
        rules = [
            ("same_name_different_business", (f["core_token_set"] >= 0.9)
             & (f["ad_jaccard"] < 0.2)),
            ("same_address_different_business", (f["ad_token_set"] >= 0.9)
             & (f["core_token_set"] < 0.5)),
        ]
    out = pd.Series("other", index=f.index)
    for name, cond in reversed(rules):              # first true condition wins
        out[cond.fillna(False).to_numpy()] = name
    return out


cat_tables = {}
for kind in ("missed", "false_merge"):
    errs = error_samples(matches["B: + M3 groups"], part, kind, n=10**9)
    cat_tables[kind] = tag(errs[[C.S1_ID, C.ENTITY_ID]], kind).value_counts()
    display(cat_tables[kind].rename(f"{kind} (arm B)").to_frame())

# 07 section 8: true pairs whose addresses share no token (a bare city on one side)
miss = error_samples(matches["B: + M3 groups"], part, "missed", n=10**9)[
    [C.S1_ID, C.ENTITY_ID]].merge(frame, on=[C.S1_ID, C.ENTITY_ID], how="left")
kept_miss = miss[miss["tok_len_l"].notna()]
no_addr = kept_miss["ad_jaccard"] == 0
misses_ad_jaccard_zero = {"kept_candidate_misses": len(kept_miss),
                          "ad_jaccard_zero": int(no_addr.sum()),
                          "median_core_token_set": float(
                              kept_miss.loc[no_addr, "core_token_set"].median())}
print("misses with no shared address token:", misses_ad_jaccard_zero)

,missed (arm B)
not a kept candidate (blocking / filter),39772
other,21092
name_typo,4737
dba_trade_name,3871
missing_address_component,366
word_order,25
postal_code_mismatch,3


,false_merge (arm B)
other,2264
same_address_different_business,445
same_name_different_business,35


misses with no shared address token: {'kept_candidate_misses': 30094, 'ad_jaccard_zero': 0, 'median_core_token_set': nan}


### 6.4 Samples of arm B's remaining false merges, with the raw records

In [13]:
def with_raw(sample: pd.DataFrame) -> pd.DataFrame:
    """Raw name / address of both sides from the Parquet cache (only the sampled ids)."""
    ids = pd.concat([sample[C.S1_ID], sample[C.ENTITY_ID]]).unique().tolist()
    raw = pd.concat([pq.read_table(C.DATASET / ".cache" / f"train_source{s}.parquet",
                                   columns=[C.ENTITY_ID, C.NAME, C.ADDRESS],
                                   filters=[(C.ENTITY_ID, "in", ids)]).to_pandas()
                     for s in C.SOURCES]).set_index(C.ENTITY_ID)
    return sample.assign(name_l=sample[C.S1_ID].map(raw[C.NAME]),
                         addr_l=sample[C.S1_ID].map(raw[C.ADDRESS]),
                         name_r=sample[C.ENTITY_ID].map(raw[C.NAME]),
                         addr_r=sample[C.ENTITY_ID].map(raw[C.ADDRESS]))


with_raw(error_samples(matches["B: + M3 groups"], part, "false_merge", n=10,
                       scored=arms["B: + M3 groups"]["scored"]))

,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-107585328,S2-474103230,0.909044,Simba International Ltd,"New Delhi, B-4/13, 2Nd Floor, Main Wali Nagar,...",Simba International,
1,S1-137695892,S3-697508841,0.890605,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
2,S1-176450615,S3-332436853,0.908510,Gurgaon India Private Limited,"Gurgaon, Dlf Building No.9, Haryana, Tower-A, ...",Gurgaon India Private Ltd,"B-03, Gurgaon, Gurugram, HR"
3,S1-357639148,S2-846895859,0.997832,West Chemical,"312 Pheasant Drive, Fl 1, Lexington, NC",West Chemical Corp,"312 PHEASANT DR, LEXINGTON, NC"
4,S1-489913517,S2-77179061,0.893023,Cornerstone Medicals Private Limited,"C/O Trimbak Gavade, House No-R. 30-02, Dhangar...",Cornerstone Mega Private Limited,"DOOR NO 99 C/O TRIMBAK GAVADE, HOUSE NO-R. 30-..."
5,S1-734849917,S3-835178366,0.889449,"Frontier Utility, LLC","Chesapeake City, Unit F, 301 Wimbledon Chase, VA","Frontier Útility, LLC",
6,S1-738128729,S2-959360977,0.988823,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
7,S1-835179861,S2-90631569,0.883551,Premier Global LLP,"505, Gf Nyay Khand-3, Indirapuram, Ghaziabad, ...",PREMIER SERVICES LLP,"509, GF NYAY KHAND-3, INDIRAPURAM, Uttar Pradesh"
8,S1-849721294,S2-256639455,0.582354,Bureau of Public Works Associates,"28 Ashmont Road, Wellesley, MA",Solarc,"12- ASHMONT ROAD, WELLESLEY, MA"
9,S1-893959950,S3-746174645,0.932484,Vikram & Partners,"3/17 Azadgarh, Kolkata, Howrah, West Bengal",Vikram Vikram Partners,


## 7. Log the result

The version's number is arm B (stage 2 with all four M3 groups); arms A and C go into
`metrics.json`. Decision against the same-machine arm A (13 §3 adapted to est_public, as v107):
KEEP if est_public(B) > est_public(A) + 0.001, DROP if not above A, else INVESTIGATE. Arm B's
two-stage pipeline is saved for test inference.

In [14]:
b, a_arm = arms["B: + M3 groups"], arms["A: v104 columns"]
est_b = float(b["res"].loc["all", "est_public"])
est_a = float(a_arm["res"].loc["all", "est_public"])
DECISION = "KEEP" if est_b > est_a + 0.001 else ("DROP" if est_b <= est_a else "INVESTIGATE")
ts = TwoStage(stage1, b["models"], b["rule"], tcfg, b["table"], {"fit": b["info"]})
ts.save(ARTIFACTS / "two_stage")
b["scored"].to_parquet(ARTIFACTS / "mock_scored_B.parquet", index=False)
a_arm["scored"].to_parquet(ARTIFACTS / "mock_scored_A.parquet", index=False)
record = {
    "hypothesis": "M3's groups on the kept pairs raise est_public over the same two-stage "
                  "pipeline without them by >= 0.001",
    "stage1": "v101", "two_stage": tcfg.record(), "m3_groups": list(M3_GROUPS),
    "rule": asdict(b["rule"]), "rule_kind": type(b["rule"]).__name__,
    **{k: float(b["res"].loc["all", k]) for k in SCORE_COLS},
    **{f"mock_{c}": float(b["res"].loc[c, "f_beta"]) for c in b["res"].index if c != "all"},
    "arms": {name: {**{k: float(x["res"].loc["all", k]) for k in SCORE_COLS},
                    "rule": asdict(x["rule"]), "rule_kind": type(x["rule"]).__name__,
                    "n_columns": len(ARMS[name]), "seconds": x["seconds"],
                    "tune_threshold": x["tune_threshold"], "tune_expected": x["tune_expected"]}
             for name, x in arms.items()},
    "parent_logged": parent_scores, "filter_report": filter_report.to_dict("index"),
    "importance_top": b["importance"].head(25).to_dict(),
    "m3_importance": m3_imp["gain"].to_dict(), "m3_group_share": group_share,
    "errors_mock": counts, "transitions": transitions,
    "error_categories": {k: v.to_dict() for k, v in cat_tables.items()},
    "misses_ad_jaccard_zero": misses_ad_jaccard_zero,
    "fit_info": b["info"], **timings, "peak_rss_gb": peak_rss_gb(), "decision": DECISION,
}
c_est = float(arms["C: + M3 without token_freq"]["res"].loc["all", "est_public"])
row = log_result(
    EXP_DIR, change="v104 two-stage + M3 groups (idf, token_freq, ctx_idf, address_extra) in "
                    "stage 2 on the kept pairs; v107 rule tuning",
    group=GROUP, mock_f05=float(b["res"].loc["all", "f_beta"]), cand_recall=None,
    notes=(f"est_public {est_b:.4f} (same-machine A {est_a:.4f}, v107 logged "
           f"{parent_scores['est_public']:.4f}); C without token_freq {c_est:.4f}"),
    metrics=record, owner=OWNER, parent="v107", decision=DECISION)
print(f"est_public A {est_a:.5f} -> B {est_b:.5f} (C {c_est:.5f}): {DECISION}")
row

est_public A 0.96588 -> B 0.96788 (C 0.96751): KEEP


{'version': 'v042',
 'date': '2026-09-26',
 'group': 'C2',
 'change': 'v104 two-stage + M3 groups (idf, token_freq, ctx_idf, address_extra) in stage 2 on the kept pairs; v107 rule tuning',
 'local_f05': '',
 'mock_f05': '0.9762',
 'cand_recall': '',
 'public_f05': '',
 'commit': '638e774',
 'notes': 'est_public 0.9679 (same-machine A 0.9659, v107 logged 0.9659); C without token_freq 0.9675',
 'owner': 'M3',
 'parent': 'v107',
 'decision': 'KEEP'}

## 8. Conclusion

* **Result (§5.2):** est_public **0.96588 → 0.96788 (+0.00200)** and mock F0.5 0.97451 →
  0.97623 (+0.00172) from adding M3's four groups to stage 2 (arm A → B). Arm A reproduces the
  logged v107 exactly (est_public 0.96588, mock F0.5 0.97451, same rule), so the gain is the
  features' alone. Singletons 0.98367 → 0.98655, pair precision 0.99648 → 0.99726, pair recall
  0.93423 → 0.93802; both countries gain (est_public India +0.0024, US +0.0016).
* **Decision (§7): KEEP** (+0.00200 > the 0.001 margin). Without `token_freq` (arm C) the gain
  is +0.00163: the raw pool counts add +0.0004 at the test's density, so their pool-size risk
  did not materialise on the mock.
* **M3 features in stage 2 (§5.3):** 2.4 % of the gain (idf 1.25 %, token_freq 0.51 %,
  address_extra 0.36 %, ctx_idf 0.26 %) next to `pool_gap` 71 % and `p1` 19 %; `freq_addr_r`
  (#7) and `idf_name_cover_r` (#8) are the best pair features of the model. Small shares, but
  they act where the competition features cannot separate two look-alikes.
* **Errors (§6):** false merges 3,522 → 2,744 (−22 %), misses 74,223 → 69,866, false singletons
  3,154 → 3,063, singleton merges 362 → 289; pairs: 1,247 false merges and 6,494 misses fixed,
  396 new false merges and 2,046 true pairs lost. Of arm B's misses, 39,772 are not kept
  candidates (blocking or the stage-1 filter: M1's side); among M3's categories `name_typo`
  (4,737) and `dba_trade_name` (3,871) lead, `missing_address_component` 366. False merges are
  mostly `other` (2,264) and `same_address_different_business` (445). No remaining candidate
  miss has two non-empty addresses without a shared token (07 §8's `ad_jaccard == 0` case).
* **Cost:** stage-1 pass 2,964 s including the mock blocking (cached for later versions), each
  arm ~8 min on the RTX 2050, peak RSS 4.2 GB. M3's groups run on the 6.3M kept pairs only.
* **Next experiment:** v043 puts M3's groups inside stage 1, where they can sharpen p1 and with
  it `pool_gap`; the better of v042 / v043 goes to M1 for test inference and an upload.


## 9. Test inference (shortlisted versions only)

With `RUN_TEST = True`: v107-style test files from arm B's saved pipeline (stage-1 outputs of
the test partitions cached per country), copied to `submissions/v042/`, then both validators.

In [15]:
if RUN_TEST:
    t0 = time.time()
    match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
        cfg, ts, cache_dir=STAGE1_CACHE / "test")
    print(f"run_test {time.time() - t0:.0f} s")
    dest = C.ROOT / "submissions" / "v042"
    dest.mkdir(parents=True, exist_ok=True)
    for p in (match_path, cand_path):
        shutil.copy2(p, dest / p.name)
    for cmd in ([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                 str(C.OUTPUT), "--check-ids"],
                [sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                 "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")]):
        out = subprocess.run(cmd, capture_output=True, text=True)
        print(out.stdout[-2000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

notebook total 4545 s, peak RSS 4.22 GB
